# Fine Tuning LLM

# Mounting drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd "your/data/path"

## Install dependencies

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q "unsloth_zoo @ git+https://github.com/unslothai/unsloth_zoo.git"
!pip install -q --no-deps "trl<0.9.0" peft accelerate bitsandbytes datasets transformers evaluate compressed-tensors

In [ ]:
try:
  import psutil
except Exception:
  psutil = None

if psutil is not None:
  dataset_num_proc = max(psutil.cpu_count()+4, 2)
else:
  dataset_num_proc = 2

### Set constants

In [ ]:
AI_ACT_QA_DATASET = "data/AI ACT/datasets/dataset_truncated.json"
GDPR_QA_DATASET = "data/GDPR/datasets/dataset_truncated.jsonl"

Example Python setup for Llama 8B (adjust model_name to match your checkpoint, e.g. unsloth/llama-3-8b-bnb-4bit or a local path).​

In [ ]:
# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # We also uploaded 4bit for 405b!
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # New Mistral 12b 2x faster!
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!
] # More models at https://huggingface.co/unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # auto
load_in_4bit = True

model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"  # or your Llama-3 8B path


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Optional: ensure chat template is set for Llama 3 style
tokenizer.padding_side = "right"
tokenizer.pad_token = tokenizer.eos_token


Add LoRA adapters (rsLoRA style recommended by Unsloth).​

In [ ]:
# Llama 3/3.1 8B (most common)
target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]

# DeepSeek / Qwen2
# target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "o_proj"]  # Extra o_proj

# GPT-NeoX / MPT
# target_modules = ["query_key_value", "dense", "dense_h_to_4h", "dense_4h_to_h"]

In [ ]:
# After model, tokenizer = FastLanguageModel.from_pretrained(...)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = target_modules,  # Or None for auto detection
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 2704,
    use_rslora = True,  # Recommended
    loftq_config = None,
)

model.print_trainable_parameters()  # Verify: ~1-2% trainable params


## Load and format your Q&A JSON

In [ ]:
import pandas as pd
from datasets import Dataset
import numpy as np

json_path = AI_ACT_QA_DATASET  # Specify the correct path to your JSONL file

# Read the JSONL file
qa_list = pd.read_json(path_or_buf=json_path, lines=True)

# Convert to ChatML-like structure
def to_chat_example(item):
    q = item["question"].strip()
    a = item["answer"].strip()
    return {
        "messages": [
            {"role": "user", "content": q},
            {"role": "assistant", "content": a},
        ]
    }

# Apply the function to each row
chat_rows = [to_chat_example(row) for _, row in qa_list.iterrows()]

np_chat_rows = np.array(chat_rows[:])
np.random.shuffle(np_chat_rows)
shuffled_chat_rows = (np_chat_rows.tolist())[:]

dataset = Dataset.from_list(shuffled_chat_rows)
print(dataset[0])


In [ ]:
# Fix: Use Hugging Face datasets' native split method instead of sklearn
train_val_test = dataset.train_test_split(test_size=0.2, seed=42)
train_val = train_val_test["train"]
test = train_val_test["test"]

train_val_split = train_val.train_test_split(test_size=0.25, seed=42)  # 0.25 of 80% = 20%
train = train_val_split["train"]
val = train_val_split["test"]

print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Now define a formatting function that turns messages into a single training string using the tokenizer’s chat template (Llama 3‑style). Unsloth’s docs recommend this pattern.​

In [ ]:
def formatting_prompts_func(examples):
    texts = []
    for msgs in examples["messages"]:
        # msgs is a list of dicts with role/content
        text = tokenizer.apply_chat_template(
            msgs,
            tokenize = False,
            add_generation_prompt = False,  # SFT: include assistant answer in labels
        )
        texts.append(text)
    return {"text": texts}

# Apply to ALL splits (train, val, test) AFTER splitting
train = train.map(formatting_prompts_func, batched=True, remove_columns=train.column_names)
val = val.map(formatting_prompts_func, batched=True, remove_columns=val.column_names)

test_raw = test
test = test.map(formatting_prompts_func, batched=True, remove_columns=test.column_names)

print(train[0]["text"][:400])


## Configure SFTTrainer with Unsloth

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"  Compute capability: {torch.cuda.get_device_capability(i)}")
        print(f"  Memory: {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")
print(f"bf16 supported: {torch.cuda.is_bf16_supported()}")


In [ ]:
import gc

gc.collect()
if torch.cuda.is_available():
  torch.cuda.empty_cache()

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback
import psutil

per_device_train_batch_size = 2
per_device_eval_batch_size = 2
gradient_accumulation_steps = 8
num_train_epochs = 100  # or use max_steps instead
learning_rate = 2e-4  # try 1e-4, 5e-5 for more conservative
warmup_ratio = 0.03
early_stopping_patience = 3

# Training with validation
training_args = TrainingArguments(
    # Output and general setup
    output_dir=".results/aiact/llama8b_unsloth_qa",
    report_to="none",

    # Batch and training configuration
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_eval_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    num_train_epochs=num_train_epochs,

    # Optimization and learning rate scheduling
    learning_rate=learning_rate,
    lr_scheduler_type="cosine",
    warmup_ratio=warmup_ratio,
    optim="adamw_torch",
    weight_decay=0.0,
    max_grad_norm=1.0,

    # GPU Precision
    bf16=False,  # Disable bf16
    fp16=True,   # Enable fp16 for T4
    # bf16=torch.cuda.is_available(),

    # Saving and evaluation strategy
    save_strategy="steps",
    eval_strategy="steps",
    save_steps=100,
    eval_steps=100,

    # Logging
    logging_steps=10,

    # Best model tracking
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train,
    eval_dataset=val,
    args=training_args,
    tokenizer=tokenizer,
    max_seq_length=2048,
    dataset_text_field="text",
    packing = True,           # pack multiple examples in one sequence (efficient)
    callbacks = [EarlyStoppingCallback(early_stopping_patience=early_stopping_patience)]
)

# Add callback
trainer.add_callback(EarlyStoppingCallback(early_stopping_patience=early_stopping_patience))

## Run training
To control training length by steps instead of epochs, you can set max_steps in TrainingArguments and omit num_train_epochs, as shown in Unsloth tuning examples.

In [ ]:
trainer.train(resume_from_checkpoint = True)

## Save and merge

In [ ]:
# Save LoRA adapters first (small)
trainer.model.save_pretrained("llama-3.2-unsloth-qa-lora-ai-act", tokenizer=tokenizer)

# Merge & save full model
trainer.model = trainer.model.merge_and_unload()  # Returns merged model
trainer.model.save_pretrained("llama-3.2-unsloth-qa-ai-act-merged")
tokenizer.save_pretrained("llama-3.2-unsloth-qa-ai-act-merged")

## Evaluation

In [ ]:
# Load LoRA for stable inference
from peft import PeftModel
base_model, tokenizer = FastLanguageModel.from_pretrained("unsloth/Llama-3.2-3B-Instruct-bnb-4bit")
model = PeftModel.from_pretrained(base_model, "llama-3.2-unsloth-qa-lora")

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"  # Sync errors for debug
import torch
from evaluate import load
import evaluate

# Load metrics (your existing code)
try:
    bleu_metric = evaluate.load("bleu")
    rouge_metric = evaluate.load("rouge")
    meteor_metric = evaluate.load("meteor")
    bertscore_metric = evaluate.load("bertscore")
except:
    print("⚠️ Installing evaluation dependencies...")
    import subprocess
    subprocess.check_call(["pip", "install", "-q", "evaluate", "sacrebleu", "rouge-score", "bert-score", "nltk"])
    bleu_metric = evaluate.load("bleu")
    rouge_metric = evaluate.load("rouge")
    meteor_metric = evaluate.load("meteor")
    bertscore_metric = evaluate.load("bertscore")

# Fix pad_token_id FIRST
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

def safe_extract_predictions(raw_dataset, model, tokenizer, max_new_tokens=64, num_samples=500):
    """Batch small samples, greedy decode (no sampling crash)"""
    predictions, references = [], []
    model.eval()

    for i, item in enumerate(raw_dataset.select(range(num_samples))):  # First 10 only
        messages = item["messages"]
        inputs = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
        ).to(model.device)

        print(f"Generating {i+1}/{num_samples}: {messages[0]['content'][:50]}...")

        with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.float16):
            # GREEDY: Avoid multinomial sampling crash
            outputs = model.generate(
                inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,  # Greedy = stable
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
            pred = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

        ref = messages[1]["content"]
        predictions.append(pred.strip())
        references.append(ref.strip())

    return predictions, references

# RUN SAFE EVAL
preds, refs = safe_extract_predictions(test_raw, model, tokenizer)
results = {
    "bleu": bleu_metric.compute(predictions=preds, references=refs),
    "rouge": rouge_metric.compute(predictions=preds, references=refs),
    "bertscore": bertscore_metric.compute(predictions=preds, references=refs, lang="en"),
    "meteor": meteor_metric.compute(predictions=preds, references=refs),
}
print("\nTest Set Results (first 500):", results)

# Show samples
for i in range(len(preds[:10])):
    print(f"\nQ: {test_raw[i]['messages'][0]['content'][:80]}...")
    print(f"Pred: {preds[i][:80]}...")
    print(f"Ref: {refs[i][:80]}...")
